In [46]:
import numpy as np

x = np.array([2, 4, 6])
y = np.array([80, 140, 190])
n = len(x)

lr = 0.01
w = 25
b = 30


y_hat = w * x + b

error = y - y_hat
mse = np.mean(error ** 2)

assert abs(mse - 66.667) < 0.001

dw = - 2 * np.mean(x * error)
db = - 2 * np.mean(error)

w -= lr * dw
b -= lr * db

assert abs(w - 25.667) < 0.001
assert abs(b - 30.133) < 0.001

In [47]:
import numpy as np

x = np.array([2, 4, 6])
y = np.array([80, 140, 190])
n = len(x)

lr = 0.06
w = 25
b = 30

mses = []

for _ in range(1000):
    y_hat = w * x + b
    error = y - y_hat
    mse = np.mean(error ** 2)
    mses.append(mse)
    dw = - 2 * np.mean(x * error)
    db = - 2 * np.mean(error)
    w -= lr * dw
    b -= lr * db

print(f"w: {w}")
print(f"b: {b}")
print(f"mse: {mse}")

w: -3.203407790959749e+128
b: -6.9150266593166955e+127
mse: 1.161875211855281e+258


In [48]:
import pandas as pd

train = pd.read_csv("../data/train_features.csv")
val = pd.read_csv("../data/val_features.csv")

y_train = train["price"]
X_train = train.drop(columns = ["price", "id"]).astype(float)

y_val = val["price"]
X_val = val.drop(columns = ["price", "id"]).astype(float)

assert len(X_train.columns) == 132
assert all(col not in X_train.columns for col in ["price", "id"])

print(X_train.max().sort_values(ascending = False).head(10))

maximum_nights          1489.00000
host_months              185.00000
amenities_count           96.00000
bedrooms                  50.00000
beds                      50.00000
latitude                  40.53553
bath_num                  38.00000
minimum_nights            32.00000
accommodates              16.00000
review_scores_rating       5.00000
dtype: float64


In [49]:
X_train_std = (X_train - X_train.mean()) / X_train.std()
X_val_std = (X_val - X_train.mean()) / X_train.std()
assert X_train_std.mean().abs().max() < 1e-9
assert abs(X_train_std.std() - 1).max() < 1e-9
print(X_val_std.mean().abs().sort_values(ascending = False).head(5))

has_reviews       0.057534
nb_San Isidro     0.056836
nb_Arapiles       0.054129
nb_Aravaca        0.052355
pt_Entire home    0.050787
dtype: float64


In [57]:
X = X_train_std.to_numpy()
y = y_train.to_numpy()
w = np.zeros(X.shape[1])
b = 0
lr = 0.1
n = len(X)

mses = []

for _ in range(10000):
    y_hat = X @ w + b
    error = y - y_hat
    mse = np.mean(error ** 2)
    mses.append(mse)
    dw = - 2 / n * X.T @ error
    db = - 2 * np.mean(error)
    w -= lr * dw
    b -= lr * db

print(f"mse: {mse}")
print(f"rmse: {mse ** 0.5}")

mse: 13774.149378412176
rmse: 117.36332211731302


In [51]:
print(y.std())

148.14030779104016


In [58]:
X_va = X_val_std.to_numpy()
y_va = y_val.to_numpy()
mse = np.mean((y_va - (X_va @ w + b)) ** 2)
rmse = mse ** 0.5
R2 = 1 - mse / (y_va.std() ** 2)

print(f"rmse: {rmse}")
print(f"R2: {R2}")

rmse: 121.27630941382458
R2: 0.3584902597607772


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

assert np.abs(w - model.coef_).max() < 1e-4
assert abs(b - model.intercept_) < 1e-8
assert abs(np.mean((y_va - model.predict(X_va)) ** 2) ** 0.5 - rmse) < 0.01